# 02 — Environmental data preparation

Preparation of WorldClim 2.1 bioclimatic predictors and extraction of environmental values for the cleaned *Sapajus libidinosus* occurrences produced by `01_occurrences.ipynb`.

This notebook currently covers the workflow developed so far:

1. locate the WorldClim BIOCLIM archive;
2. verify the 19 BIOCLIM layers;
3. extract BIO1, BIO4, BIO12 and BIO15;
4. inspect raster metadata without loading the global raster unnecessarily;
5. visualize a Brazil-scale window;
6. load the 606 unique occurrence coordinates from notebook 01;
7. extract the four climatic predictors for every occurrence;
8. identify environmental NoData records;
9. remove only occurrences lacking the required environmental predictors;
10. quantify redundancy at the WorldClim raster-cell level.

Elevation, calibration area/background, coordinate-uncertainty filtering, spatial thinning and model training are deliberately left for later stages.


In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds, Window
import matplotlib.pyplot as plt

# =========================
# PROJECT PATHS
# =========================
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
WORLDCLIM_DIR = RAW_DIR / "worldclim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ENV_DIR = PROCESSED_DIR / "environment"

OCCURRENCE_FILE = PROCESSED_DIR / "sapajus_occurrences_unique.gpkg"
AUDIT_01_FILE = PROCESSED_DIR / "audit_01_occurrences.csv"

ENV_DIR.mkdir(parents=True, exist_ok=True)

print("WorldClim:", WORLDCLIM_DIR.resolve())
print("Processed:", PROCESSED_DIR.resolve())
print("Environment:", ENV_DIR.resolve())


## 1. Locate and validate the WorldClim BIOCLIM archive


In [ ]:
bio_zips = sorted(WORLDCLIM_DIR.glob("*bio*.zip"))

print(f"BIOCLIM ZIP candidates: {len(bio_zips)}")
for path in bio_zips:
    print(" -", path.name)

if len(bio_zips) != 1:
    raise RuntimeError(
        f"Expected exactly 1 BIOCLIM ZIP in {WORLDCLIM_DIR}, "
        f"but found {len(bio_zips)}."
    )

bio_zip = bio_zips[0]
print("\nSelected:", bio_zip.name)


In [ ]:
with zipfile.ZipFile(bio_zip, "r") as z:
    bio_files = sorted(
        name for name in z.namelist()
        if name.lower().endswith(".tif")
    )

print("BIOCLIM rasters found:", len(bio_files))

if len(bio_files) != 19:
    raise RuntimeError(
        f"Expected 19 BIOCLIM GeoTIFFs, found {len(bio_files)}."
    )

for name in bio_files:
    print(name)


## 2. Extract the predictors selected for the first SDM prototype

The current prototype uses:

- **BIO1** — Annual Mean Temperature
- **BIO4** — Temperature Seasonality
- **BIO12** — Annual Precipitation
- **BIO15** — Precipitation Seasonality

This is a provisional predictor set for development of the pipeline, not a claim that these four variables constitute the final ecological predictor set.


In [ ]:
SELECTED_BIO = {
    "BIO1": 1,
    "BIO4": 4,
    "BIO12": 12,
    "BIO15": 15
}

with zipfile.ZipFile(bio_zip, "r") as z:
    names = set(z.namelist())

    for variable, number in SELECTED_BIO.items():
        filename = f"wc2.1_30s_bio_{number}.tif"

        if filename not in names:
            raise FileNotFoundError(
                f"{filename} was not found inside {bio_zip.name}."
            )

        output = ENV_DIR / filename

        # Re-running the notebook remains safe: extraction overwrites
        # the same deterministic output file.
        with z.open(filename) as src, open(output, "wb") as dst:
            dst.write(src.read())

        print(f"{variable}: {output.name}")


In [ ]:
BIO_PATHS = {
    variable: ENV_DIR / f"wc2.1_30s_bio_{number}.tif"
    for variable, number in SELECTED_BIO.items()
}

for variable, path in BIO_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(path)

bio1_path = BIO_PATHS["BIO1"]
print("Environmental rasters: OK")


## 3. Inspect raster metadata


In [ ]:
with rasterio.open(bio1_path) as src:
    raster_metadata = {
        "CRS": str(src.crs),
        "width": src.width,
        "height": src.height,
        "resolution_x": src.res[0],
        "resolution_y": src.res[1],
        "bounds": src.bounds,
        "nodata": src.nodata,
        "dtype": src.dtypes[0]
    }

display(pd.Series(raster_metadata, name="BIO1"))


The WorldClim layer is global (43,200 × 21,600 cells at 30 arc-seconds).  
The notebook therefore avoids rendering the entire raster as a full RGBA image, which previously caused a very large memory allocation. Raster windows are used instead.


## 4. Inspect a Brazil-scale raster window


In [ ]:
# Approximate geographic window used only for visual inspection.
# It does not define the SDM calibration area.
BRAZIL_WINDOW = {
    "left": -74,
    "bottom": -34,
    "right": -34,
    "top": 6
}

with rasterio.open(bio1_path) as src:
    window = from_bounds(
        BRAZIL_WINDOW["left"],
        BRAZIL_WINDOW["bottom"],
        BRAZIL_WINDOW["right"],
        BRAZIL_WINDOW["top"],
        transform=src.transform
    )

    bio1_br = src.read(
        1,
        window=window,
        masked=True
    )

print("Window shape:", bio1_br.shape)
print("Minimum:", float(bio1_br.min()))
print("Maximum:", float(bio1_br.max()))
print("Mean:", float(bio1_br.mean()))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

image = ax.imshow(
    bio1_br,
    extent=[
        BRAZIL_WINDOW["left"],
        BRAZIL_WINDOW["right"],
        BRAZIL_WINDOW["bottom"],
        BRAZIL_WINDOW["top"]
    ],
    origin="upper"
)

ax.set_title("WorldClim BIO1 — Annual Mean Temperature")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

fig.colorbar(image, ax=ax, label="Temperature (°C)")
plt.show()


## 5. Load the cleaned occurrence dataset from notebook 01


In [ ]:
if not OCCURRENCE_FILE.exists():
    raise FileNotFoundError(
        f"{OCCURRENCE_FILE} does not exist. "
        "Run 01_occurrences.ipynb first."
    )

occ = gpd.read_file(OCCURRENCE_FILE)

print("Occurrences:", len(occ))
print("CRS:", occ.crs)

if occ.crs is None:
    raise ValueError("Occurrence dataset has no CRS.")

if occ.crs.to_epsg() != 4326:
    occ = occ.to_crs("EPSG:4326")
    print("Occurrences reprojected to EPSG:4326.")

assert len(occ) == 606, (
    f"Expected 606 unique occurrence coordinates, got {len(occ)}."
)

display(occ.head())


## 6. Demonstrate point-to-raster extraction


In [ ]:
# First occurrence in the cleaned dataset
example = occ.iloc[0]

lon = float(example["decimalLongitude"])
lat = float(example["decimalLatitude"])

with rasterio.open(bio1_path) as src:
    row, col = src.index(lon, lat)
    sample = next(src.sample([(lon, lat)], masked=True))[0]

    bio1_example = (
        np.nan if np.ma.is_masked(sample)
        else float(sample)
    )

print("Longitude:", lon)
print("Latitude:", lat)
print("Raster row:", row)
print("Raster column:", col)
print("BIO1:", bio1_example)


## 7. Extract BIO1, BIO4, BIO12 and BIO15 for all occurrences


In [ ]:
ENV_VARS = list(BIO_PATHS.keys())

coords = list(
    zip(
        occ["decimalLongitude"].astype(float),
        occ["decimalLatitude"].astype(float)
    )
)

for variable, path in BIO_PATHS.items():
    values = []

    with rasterio.open(path) as src:
        for sample in src.sample(coords, masked=True):
            value = sample[0]

            if np.ma.is_masked(value):
                values.append(np.nan)
            else:
                values.append(float(value))

    occ[variable] = np.asarray(values, dtype="float64")

print("Environmental column dtypes:")
display(occ[ENV_VARS].dtypes.to_frame("dtype"))


## 8. Audit extracted environmental values


In [ ]:
nodata_by_variable = occ[ENV_VARS].isna().sum()

print("NoData by predictor:")
display(nodata_by_variable.to_frame("n"))

print("\nEnvironmental summary:")
display(occ[ENV_VARS].describe().T)


In [ ]:
nodata_mask = occ[ENV_VARS].isna().any(axis=1)

nodata_occ = occ.loc[
    nodata_mask,
    [
        "gbifID",
        "species",
        "decimalLatitude",
        "decimalLongitude",
        "stateProvince",
        "locality",
        "coordinateUncertaintyInMeters",
        "year",
        "basisOfRecord",
        "issue"
    ]
].copy()

print(
    "Occurrences with at least one missing environmental predictor:",
    int(nodata_mask.sum())
)

display(nodata_occ)


### Inspect the local raster neighborhood of any NoData occurrence

This diagnostic does **not** move or modify the occurrence coordinate.  
It only helps determine whether a missing environmental value occurs at a local raster boundary such as a coastal or water cell.


In [ ]:
if nodata_mask.sum() > 0:
    first_nodata = occ.loc[nodata_mask].iloc[0]

    lon = float(first_nodata["decimalLongitude"])
    lat = float(first_nodata["decimalLatitude"])

    with rasterio.open(bio1_path) as src:
        row, col = src.index(lon, lat)

        neighborhood_window = Window(
            col - 2,
            row - 2,
            5,
            5
        )

        neighborhood = src.read(
            1,
            window=neighborhood_window,
            masked=True
        )

    print("NoData occurrence:")
    print("gbifID:", first_nodata["gbifID"])
    print("Longitude:", lon)
    print("Latitude:", lat)
    print("Row:", row)
    print("Column:", col)
    print("\nBIO1 neighborhood (5 × 5):")
    print(neighborhood)
else:
    print("No environmental NoData occurrences detected.")


## 9. Create the complete-environment occurrence dataset

Occurrences lacking any of the required environmental predictors are excluded from the **modeling dataset**, while the original occurrence dataset from notebook 01 remains unchanged.


In [ ]:
occ_env = occ.loc[~nodata_mask].copy().reset_index(drop=True)

print("Occurrences before environmental NoData filter:", len(occ))
print("Removed at this step:", int(nodata_mask.sum()))
print("Occurrences with complete climate data:", len(occ_env))

assert len(occ_env) == 605, (
    f"Expected 605 occurrences with complete climate data, "
    f"got {len(occ_env)}."
)


## 10. Quantify redundancy at the WorldClim cell level


In [ ]:
with rasterio.open(bio1_path) as src:
    cells = [
        src.index(float(lon), float(lat))
        for lon, lat in zip(
            occ_env["decimalLongitude"],
            occ_env["decimalLatitude"]
        )
    ]

occ_env["raster_row"] = [row for row, col in cells]
occ_env["raster_col"] = [col for row, col in cells]

n_records = len(occ_env)

n_unique_cells = (
    occ_env[["raster_row", "raster_col"]]
    .drop_duplicates()
    .shape[0]
)

n_redundant_cells = n_records - n_unique_cells

print("Occurrences with complete climate data:", n_records)
print("Unique WorldClim cells:", n_unique_cells)
print("Occurrences redundant at raster-cell level:", n_redundant_cells)
print(
    "Raster-cell redundancy:",
    f"{(n_redundant_cells / n_records * 100):.2f}%"
)


In [ ]:
cell_counts = (
    occ_env
    .groupby(["raster_row", "raster_col"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

display(cell_counts.head(20))


## 11. Build a cumulative audit log


In [ ]:
if AUDIT_01_FILE.exists():
    audit_01 = pd.read_csv(AUDIT_01_FILE)
else:
    # Fallback keeps the notebook understandable, but notebook 01 should
    # normally have produced the audit file.
    audit_01 = pd.DataFrame([
        {
            "step": "GBIF raw download",
            "n_records": 728,
            "removed_at_step": 0
        },
        {
            "step": "Valid geographic coordinates",
            "n_records": 728,
            "removed_at_step": 0
        },
        {
            "step": "Unique exact coordinates",
            "n_records": len(occ),
            "removed_at_step": 122
        }
    ])

audit_02 = pd.concat(
    [
        audit_01,
        pd.DataFrame([
            {
                "step": "Complete BIO1/BIO4/BIO12/BIO15 data",
                "n_records": len(occ_env),
                "removed_at_step": int(nodata_mask.sum())
            }
        ])
    ],
    ignore_index=True
)

display(audit_02)


## 12. Export the current environmental-stage products

The dataset exported here still contains all occurrences occupying the same WorldClim cell.  
Cell-level deduplication is **diagnosed but not yet applied**, because it should be considered together with coordinate uncertainty, predictor resolution and the subsequent SDM design.


In [ ]:
environment_output = (
    PROCESSED_DIR /
    "sapajus_occurrences_environment.gpkg"
)

audit_output = (
    PROCESSED_DIR /
    "audit_02_environment.csv"
)

occ_env.to_file(
    environment_output,
    driver="GPKG"
)

audit_02.to_csv(
    audit_output,
    index=False
)

print("Saved environmental occurrence dataset:")
print(environment_output.resolve())

print("\nSaved cumulative audit log:")
print(audit_output.resolve())


In [ ]:
uncertainty = occ_env["coordinateUncertaintyInMeters"]

print("=== COORDINATE UNCERTAINTY ===")
print("Occurrences:", len(occ_env))
print("Known uncertainty:", uncertainty.notna().sum())
print("Unknown uncertainty:", uncertainty.isna().sum())

thresholds = [100, 500, 1000, 5000, 10000, 50000]

print("\nKnown uncertainty thresholds:")

for threshold in thresholds:
    n = (uncertainty <= threshold).sum()
    pct_known = n / uncertainty.notna().sum() * 100
    pct_total = n / len(occ_env) * 100

    print(
        f"≤ {threshold:>5} m: "
        f"{n:>3} | "
        f"{pct_known:5.1f}% of known | "
        f"{pct_total:5.1f}% of total"
    )

In [ ]:
pixel_uncertainty = (
    occ_env
    .groupby(["raster_row", "raster_col"])
    .agg(
        n_records=("gbifID", "size"),
        n_known_uncertainty=(
            "coordinateUncertaintyInMeters",
            lambda x: x.notna().sum()
        ),
        min_uncertainty=(
            "coordinateUncertaintyInMeters",
            "min"
        ),
        median_uncertainty=(
            "coordinateUncertaintyInMeters",
            "median"
        )
    )
    .reset_index()
)

display(
    pixel_uncertainty
    .sort_values("n_records", ascending=False)
    .head(20)
)

In [ ]:
thresholds = [1000, 5000, 10000, 50000]

results = []

for threshold in thresholds:

    # Mantém:
    # 1. incerteza conhecida <= limiar
    # 2. incerteza desconhecida
    keep = (
        occ_env["coordinateUncertaintyInMeters"].isna()
        |
        (
            occ_env["coordinateUncertaintyInMeters"]
            <= threshold
        )
    )

    temp = occ_env.loc[keep].copy()

    n_records = len(temp)

    n_cells = (
        temp[["raster_row", "raster_col"]]
        .drop_duplicates()
        .shape[0]
    )

    results.append({
        "threshold_m": threshold,
        "records_retained": n_records,
        "records_removed": len(occ_env) - n_records,
        "unique_cells": n_cells,
        "records_per_unique_cell": n_records / n_cells
    })

sensitivity = pd.DataFrame(results)

display(sensitivity)

In [ ]:
def uncertainty_class(x):
    if pd.isna(x):
        return "Unknown"
    elif x <= 1000:
        return "≤ 1 km"
    elif x <= 5000:
        return "1–5 km"
    elif x <= 10000:
        return "5–10 km"
    elif x <= 50000:
        return "10–50 km"
    else:
        return "> 50 km"


occ_env["uncertainty_class"] = (
    occ_env["coordinateUncertaintyInMeters"]
    .apply(uncertainty_class)
)

occ_env["uncertainty_class"].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

classes = [
    "≤ 1 km",
    "1–5 km",
    "5–10 km",
    "10–50 km",
    "> 50 km",
    "Unknown"
]

for category in classes:

    subset = occ_env[
        occ_env["uncertainty_class"] == category
    ]

    ax.scatter(
        subset["decimalLongitude"],
        subset["decimalLatitude"],
        label=f"{category} (n={len(subset)})",
        s=20,
        alpha=0.7
    )

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    "Sapajus libidinosus — coordinate uncertainty"
)

ax.legend(
    title="Coordinate uncertainty",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
high_uncertainty = (
    occ_env[
        occ_env["coordinateUncertaintyInMeters"] > 10000
    ]
    [
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "coordinateUncertaintyInMeters",
            "year",
            "basisOfRecord",
            "institutionCode",
            "datasetKey",
            "stateProvince",
            "issue"
        ]
    ]
    .sort_values(
        "coordinateUncertaintyInMeters",
        ascending=False
    )
)

print("Records > 10 km:", len(high_uncertainty))

display(high_uncertainty)

In [ ]:
display(
    occ_env[
        occ_env["coordinateUncertaintyInMeters"] > 10000
    ]
    .groupby("institutionCode", dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("n")
    .to_frame()
)

In [ ]:
# ============================================================
# FILTER COORDINATE UNCERTAINTY
# ============================================================

uncertainty_limit = 10_000  # metres

known_uncertainty = occ_env["coordinateUncertaintyInMeters"].notna()

occ_filtered = occ_env[
    (~known_uncertainty) |
    (occ_env["coordinateUncertaintyInMeters"] <= uncertainty_limit)
].copy()

removed_uncertainty = occ_env[
    known_uncertainty &
    (occ_env["coordinateUncertaintyInMeters"] > uncertainty_limit)
].copy()

print("Before uncertainty filter:", len(occ_env))
print("After uncertainty filter:", len(occ_filtered))
print("Removed:", len(removed_uncertainty))
print(
    "Unknown uncertainty retained:",
    occ_filtered["coordinateUncertaintyInMeters"].isna().sum()
)

print(
    "Unique WorldClim cells retained:",
    occ_filtered[
        ["raster_row", "raster_col"]
    ].drop_duplicates().shape[0]
)

In [ ]:
# ============================================================
# MAP AFTER COORDINATE-UNCERTAINTY FILTER
# ============================================================

fig, ax = plt.subplots(figsize=(8, 10))

ax.scatter(
    occ_filtered["decimalLongitude"],
    occ_filtered["decimalLatitude"],
    s=18,
    alpha=0.65
)

ax.set_title(
    "Sapajus libidinosus — occurrences after uncertainty filter"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.grid(alpha=0.2)

plt.show()

In [ ]:
# ============================================================
# WESTERNMOST OCCURRENCES AFTER FILTER
# ============================================================

westernmost = (
    occ_filtered[
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "coordinateUncertaintyInMeters",
            "year",
            "basisOfRecord",
            "institutionCode",
            "stateProvince",
            "issue"
        ]
    ]
    .sort_values("decimalLongitude")
    .head(10)
)

westernmost

In [ ]:
# ============================================================
# FLAG GEOGRAPHIC OUTLIERS FOR MANUAL REVIEW
# ============================================================

# Western records selected only for inspection.
# This is NOT an exclusion criterion.

geo_review = (
    occ_filtered.loc[
        occ_filtered["decimalLongitude"] < -55,
        [
            "gbifID",
            "species",
            "decimalLongitude",
            "decimalLatitude",
            "coordinateUncertaintyInMeters",
            "year",
            "basisOfRecord",
            "institutionCode",
            "stateProvince",
            "locality",
            "datasetKey",
            "issue"
        ]
    ]
    .sort_values("decimalLongitude")
    .copy()
)

print("Records flagged for geographic review:", len(geo_review))

geo_review

In [ ]:
# ============================================================
# SUMMARY OF WESTERN RECORDS
# ============================================================

print("States:")
print(
    geo_review["stateProvince"]
    .value_counts(dropna=False)
)

print("\nInstitutions:")
print(
    geo_review["institutionCode"]
    .value_counts(dropna=False)
)

print("\nYears:")
print(
    geo_review["year"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# ============================================================
# QUALITY-CONTROL FLAGS
# ============================================================

occ_qc = occ_filtered.copy()

# Coordinate uncertainty
occ_qc["qc_uncertainty"] = "unknown"

known = occ_qc["coordinateUncertaintyInMeters"].notna()

occ_qc.loc[
    known &
    (occ_qc["coordinateUncertaintyInMeters"] <= 10_000),
    "qc_uncertainty"
] = "acceptable"

# Geographic/taxonomic review
occ_qc["qc_geographic_outlier"] = False

# Amazonas record identified during geographic review
occ_qc.loc[
    occ_qc["gbifID"].astype(str) == "6510135994",
    "qc_geographic_outlier"
] = True

# Do NOT exclude automatically yet
occ_qc["qc_keep"] = True

print("Total records:", len(occ_qc))
print("\nUncertainty QC:")
print(occ_qc["qc_uncertainty"].value_counts())

print("\nGeographic outliers flagged:")
print(occ_qc["qc_geographic_outlier"].value_counts())

print("\nRecords currently retained:")
print(occ_qc["qc_keep"].value_counts())

In [ ]:
# ============================================================
# SPATIAL OUTLIER DIAGNOSTIC
# Nearest-neighbour distance
# ============================================================

from sklearn.neighbors import BallTree

# Coordinates in radians for haversine distance
coords_rad = np.radians(
    occ_qc[
        ["decimalLatitude", "decimalLongitude"]
    ].to_numpy()
)

tree = BallTree(
    coords_rad,
    metric="haversine"
)

# k=2 because the nearest point is the record itself
distances, indices = tree.query(
    coords_rad,
    k=2
)

EARTH_RADIUS_KM = 6371.0088

occ_qc["nearest_neighbor_km"] = (
    distances[:, 1] * EARTH_RADIUS_KM
)

print("Nearest-neighbour distance (km):")

display(
    occ_qc["nearest_neighbor_km"]
    .describe()
)

In [ ]:
# ============================================================
# MOST SPATIALLY ISOLATED RECORDS
# ============================================================

spatial_outliers = (
    occ_qc[
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "nearest_neighbor_km",
            "coordinateUncertaintyInMeters",
            "year",
            "institutionCode",
            "stateProvince",
            "qc_uncertainty",
            "qc_geographic_outlier"
        ]
    ]
    .sort_values(
        "nearest_neighbor_km",
        ascending=False
    )
    .head(20)
)

display(spatial_outliers)

In [ ]:
# ============================================================
# AUTOMATIC SPATIAL-ISOLATION FLAG
# ============================================================

q1 = occ_qc["nearest_neighbor_km"].quantile(0.25)
q3 = occ_qc["nearest_neighbor_km"].quantile(0.75)

iqr = q3 - q1

isolation_threshold = q3 + 1.5 * iqr

occ_qc["qc_spatial_isolation"] = (
    occ_qc["nearest_neighbor_km"]
    > isolation_threshold
)

print(f"Q1: {q1:.2f} km")
print(f"Q3: {q3:.2f} km")
print(f"IQR: {iqr:.2f} km")
print(
    f"Exploratory isolation threshold: "
    f"{isolation_threshold:.2f} km"
)

print("\nFlagged records:")
print(
    occ_qc["qc_spatial_isolation"]
    .value_counts()
)

In [ ]:
# ============================================================
# BIOGEOGRAPHIC QC
# ============================================================

reviewed_outliers = [
    "6510135994",  # Amazonas
    "6510093434",  # Santa Catarina
]

occ_qc["qc_geographic_outlier"] = (
    occ_qc["gbifID"]
    .astype(str)
    .isin(reviewed_outliers)
)

occ_qc["qc_outlier_reason"] = ""

occ_qc.loc[
    occ_qc["gbifID"].astype(str) == "6510135994",
    "qc_outlier_reason"
] = "Outside documented native range — Amazonas"

occ_qc.loc[
    occ_qc["gbifID"].astype(str) == "6510093434",
    "qc_outlier_reason"
] = "Outside documented native range — Santa Catarina"

display(
    occ_qc.loc[
        occ_qc["qc_geographic_outlier"],
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "nearest_neighbor_km",
            "qc_outlier_reason"
        ]
    ]
)

In [ ]:
# ============================================================
# FINAL QC DECISION FOR REVIEWED GEOGRAPHIC OUTLIERS
# ============================================================

occ_qc["qc_keep"] = ~occ_qc["qc_geographic_outlier"]

print("QC dataset:", len(occ_qc))
print("Retained for modeling:", occ_qc["qc_keep"].sum())
print("Excluded after geographic review:", (~occ_qc["qc_keep"]).sum())

In [ ]:
occ_model = (
    occ_qc.loc[occ_qc["qc_keep"]]
    .copy()
    .reset_index(drop=True)
)

print("Modeling occurrences:", len(occ_model))

In [ ]:
n_unique_cells_qc = (
    occ_model[
        ["raster_row", "raster_col"]
    ]
    .drop_duplicates()
    .shape[0]
)

n_cell_redundant = len(occ_model) - n_unique_cells_qc

print("Occurrences after QC:", len(occ_model))
print("Unique WorldClim cells:", n_unique_cells_qc)
print("Redundant occurrences:", n_cell_redundant)
print(
    "Raster-cell redundancy:",
    f"{n_cell_redundant / len(occ_model) * 100:.2f}%"
)

In [ ]:
# ============================================================
# ENVIRONMENTAL-GRID THINNING
# One presence per WorldClim cell
# ============================================================

occ_model["_uncertainty_sort"] = (
    occ_model["coordinateUncertaintyInMeters"]
    .fillna(np.inf)
)

occ_model["_gbif_sort"] = (
    occ_model["gbifID"]
    .astype(str)
)

occ_thinned = (
    occ_model
    .sort_values(
        [
            "raster_row",
            "raster_col",
            "_uncertainty_sort",
            "_gbif_sort"
        ],
        kind="stable"
    )
    .drop_duplicates(
        subset=["raster_row", "raster_col"],
        keep="first"
    )
    .drop(
        columns=[
            "_uncertainty_sort",
            "_gbif_sort"
        ]
    )
    .reset_index(drop=True)
)

print("Before raster-cell thinning:", len(occ_model))
print("After raster-cell thinning:", len(occ_thinned))
print("Removed as cell-level redundancy:",
      len(occ_model) - len(occ_thinned))

In [ ]:
print("=== THINNED DATASET ===")

print("Occurrences:", len(occ_thinned))

print(
    "Known uncertainty:",
    occ_thinned[
        "coordinateUncertaintyInMeters"
    ].notna().sum()
)

print(
    "Unknown uncertainty:",
    occ_thinned[
        "coordinateUncertaintyInMeters"
    ].isna().sum()
)

print("\nUncertainty summary:")

display(
    occ_thinned[
        "coordinateUncertaintyInMeters"
    ].describe()
)

In [ ]:
assert not occ_thinned.duplicated(
    subset=["raster_row", "raster_col"]
).any()

assert len(occ_thinned) == 349

print("One presence per WorldClim cell: OK")

In [ ]:
# ============================================================
# SAVE FINAL PRESENCE DATASET AND UPDATE AUDIT LOG
# ============================================================

# ------------------------------------------------------------
# 1. Save final modeling occurrences
# ------------------------------------------------------------

output_presence = (
    PROCESSED_DIR /
    "sapajus_occurrences_modeling.gpkg"
)

occ_thinned.to_file(
    output_presence,
    driver="GPKG"
)

print("Saved modeling occurrences:")
print(output_presence)


# ------------------------------------------------------------
# 2. Load existing audit log from disk
# ------------------------------------------------------------

audit_path = (
    PROCESSED_DIR /
    "audit_02_environment.csv"
)

if not audit_path.exists():
    raise FileNotFoundError(
        f"Audit log not found: {audit_path}"
    )

audit_previous = pd.read_csv(audit_path)

print("\nExisting audit log:")
display(audit_previous)


# ------------------------------------------------------------
# 3. Add current QC and thinning steps
# ------------------------------------------------------------

new_steps = pd.DataFrame([
    {
        "step": "Coordinate uncertainty filter",
        "n_records": 555,
        "removed_at_step": 50,
        "description": (
            "Removed records with known coordinate uncertainty "
            "> 10 km; records with unknown uncertainty retained."
        )
    },
    {
        "step": "Biogeographic QC",
        "n_records": 553,
        "removed_at_step": 2,
        "description": (
            "Excluded two manually reviewed geographic/taxonomic "
            "outliers outside the documented native distribution."
        )
    },
    {
        "step": "WorldClim cell thinning",
        "n_records": len(occ_thinned),
        "removed_at_step": len(occ_model) - len(occ_thinned),
        "description": (
            "Retained one presence per WorldClim 30 arc-second cell, "
            "prioritizing the record with the lowest known coordinate "
            "uncertainty."
        )
    }
])


# ------------------------------------------------------------
# 4. Prevent duplicated audit entries if cell is rerun
# ------------------------------------------------------------

steps_to_replace = new_steps["step"].tolist()

audit_previous = audit_previous[
    ~audit_previous["step"].isin(steps_to_replace)
].copy()

audit_final = pd.concat(
    [audit_previous, new_steps],
    ignore_index=True
)


# ------------------------------------------------------------
# 5. Save updated audit log
# ------------------------------------------------------------

audit_final.to_csv(
    audit_path,
    index=False
)

print("\nUpdated audit log:")
display(audit_final)

print("\nFinal modeling dataset:", len(occ_thinned))
print(
    "Unique WorldClim cells:",
    occ_thinned[
        ["raster_row", "raster_col"]
    ].drop_duplicates().shape[0]
)

## Current stopping point

At this stage the pipeline should reproduce:

**728 raw GBIF records → 606 unique exact coordinates → 605 occurrences with complete BIO1/BIO4/BIO12/BIO15 data**

The next methodological decisions are intentionally not implemented yet:

- coordinate-uncertainty threshold;
- treatment of multiple occurrences in the same environmental cell;
- elevation;
- final predictor selection/collinearity assessment;
- accessible/calibration area (M);
- background or pseudoabsence sampling;
- spatial thinning;
- spatial train/test partitioning;
- feature scaling;
- MLP/DNN and benchmark SDMs.
